# FLUX Phase 4: Thermodynamic Learning (TL)
## Complete Pipeline — Train, Test, Demo, Upload

> **Goal:** Replace backpropagation with local thermodynamic settling. The system learns from single examples in real time without a training loop. Inference and learning are the same operation.

### What this notebook does:
1. **Setup** — Clone/pull repo, install deps, detect hardware
2. **Load Phases 1 & 2** — Verify CSE + ResonanceField checkpoints
3. **Smoke Test** — Verify ThermodynamicLearner builds and runs
4. **Train** — One-shot fact learning, retention test, temperature annealing, SGD comparison
5. **Upload** — Checkpoint → HuggingFace Hub
6. **Test 1** — Single-shot learning retention (learned fact survives 100+ updates)
7. **Test 2** — No global gradient required (verified zero .grad)
8. **Test 3** — Temperature annealing behavior (correct heating/cooling)
9. **Demo 1** — One-shot fact learning visualization
10. **Demo 2** — Compare thermodynamic learning to SGD convergence
11. **Interactive** — Learn custom facts, query them back
12. **Results** — View RESULTS_PHASE_4.md + full log
13. **Final** — Upload logs → HuggingFace & push to GitHub
14. **Artifacts** — Save checkpoint, logs, results to Kaggle output

### Key Physics:
- `settling = learning` — energy minimization IS the forward+backward pass combined
- `temperature` — controls plasticity: high temp = fluid (learns fast), low temp = rigid (stable)
- `surprise → heating` — prediction errors heat the field locally, making it receptive
- `no gradients` — all updates are local perturbations, not backpropagated gradients
- `no epochs` — continuous stream, each sample processed exactly once

### Secrets Required:
- **`HF_TOKEN`** — HuggingFace write token (Add via Kaggle → Add-ons → Secrets)

---
## Cell 1: Clone / Pull Repository

In [ ]:
import os
import sys
import subprocess
import importlib
from pathlib import Path

REPO_URL = "https://github.com/Unseengap/FLUX.git"
WORK_DIR = Path("/kaggle/working/FLUX")

# ─────────────────────────────────────────────
# 1. Clone or Pull — always get the absolute
#    latest code from GitHub.
# ─────────────────────────────────────────────
if WORK_DIR.exists() and (WORK_DIR / '.git').exists():
    print("ℹ Repo already exists — pulling latest changes...")

    subprocess.run(
        ['git', 'remote', 'set-url', 'origin', REPO_URL],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )

    subprocess.run(['git', 'checkout', '.'], cwd=str(WORK_DIR),
                   capture_output=True, text=True)
    subprocess.run(['git', 'clean', '-fd'], cwd=str(WORK_DIR),
                   capture_output=True, text=True)

    fetch = subprocess.run(['git', 'fetch', '--all'], cwd=str(WORK_DIR),
                           capture_output=True, text=True)
    if fetch.returncode != 0:
        print(f"  ⚠ git fetch failed: {fetch.stderr.strip()}")
    result = subprocess.run(
        ['git', 'reset', '--hard', 'origin/main'],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )
    print(result.stdout.strip() or result.stderr.strip())

    head = subprocess.run(
        ['git', 'log', '--oneline', '-3'],
        cwd=str(WORK_DIR), capture_output=True, text=True,
    )
    print(f"\n  Latest commits:\n{head.stdout.strip()}")
    print("\n✓ Pulled & reset to latest origin/main")
else:
    if WORK_DIR.exists():
        import shutil
        shutil.rmtree(str(WORK_DIR))
    print(f"ℹ Cloning {REPO_URL}...")
    subprocess.run(['git', 'clone', REPO_URL, str(WORK_DIR)], check=True)
    print("✓ Cloned successfully")

os.chdir(str(WORK_DIR))
print(f"\nWorking directory: {os.getcwd()}")

# ─────────────────────────────────────────────
# 2. Flush cached Python modules so the kernel
#    picks up the freshly-pulled code, not stale
#    imports from a previous cell run.
# ─────────────────────────────────────────────
_stale = [m for m in sys.modules if any(
    m.startswith(p) for p in (
        'thermodynamic', 'temperature', 'energy_functions', 'online_learner',
        'cse', 'field', 'flux_utils', 'wave_types', 'interference',
        'attractor', 'field_ops', 'train_', 'demo_', 'test_',
    )
)]
for m in _stale:
    del sys.modules[m]
if _stale:
    print(f"  ✓ Flushed {len(_stale)} cached modules: {_stale[:5]}{'...' if len(_stale) > 5 else ''}")

# ─────────────────────────────────────────────
# 3. Delete stale Phase 4 checkpoint so training
#    runs fresh with the updated code.
# ─────────────────────────────────────────────
_stale_ckpt = WORK_DIR / 'checkpoints' / 'phase4.phase.pt'
if _stale_ckpt.exists():
    _stale_ckpt.unlink()
    print(f"  ✓ Deleted stale checkpoint: {_stale_ckpt.name}")

subprocess.run(['python', 'setup.py'], cwd=str(WORK_DIR),
               capture_output=True, text=True)

# Quick sanity check: verify Phase 4 modules are present
_tl_path = WORK_DIR / 'phases' / 'phase4' / 'thermodynamic.py'
_tl_text = _tl_path.read_text()
assert 'ThermodynamicLearner' in _tl_text, "ERROR: thermodynamic.py missing ThermodynamicLearner!"
assert 'settle_once' in _tl_text, "ERROR: thermodynamic.py missing settle_once!"
print(f"  ✓ thermodynamic.py verified: ThermodynamicLearner + settle_once present")

print("✓ setup.py refreshed")

---
## Cell 2: Install Dependencies & Setup

In [ ]:
!pip install -q datasets rich faiss-cpu huggingface_hub
!python setup.py

---
## Cell 3: Initialize Logger, Detect Hardware & Load Secrets

In [ ]:
import sys, torch, platform, importlib
from pathlib import Path

# ── Add project paths ──
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase1'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase2'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase3'))
sys.path.insert(0, str(Path.cwd() / 'phases' / 'phase4'))

# ── Force-reload all project modules ──
for _mod_name in list(sys.modules.keys()):
    if any(_mod_name.startswith(p) for p in (
        'flux_utils', 'thermodynamic', 'temperature', 'energy_functions',
        'online_learner', 'cse', 'field', 'wave_types', 'interference',
        'attractor', 'field_ops',
    )):
        del sys.modules[_mod_name]

from flux_utils import (
    get_device, get_hardware_info, PhaseLogger, _resolve_hf_token,
    load_checkpoint, save_checkpoint, verify_checkpoint_chain,
    upload_checkpoint_to_hf, upload_logs_to_hf, git_commit_and_push,
    PhaseResults, checkpoint_exists,
)

# ── Initialize Phase 4 Logger ──
log = PhaseLogger(phase=4)
log.separator("Phase 4: Thermodynamic Learning")
log.cell_start("Cell 3 — Hardware & Secrets")

# ── Detect hardware ──
DEVICE = get_device()
hw = get_hardware_info()
log.info(f"Device: {DEVICE}")
log.info(f"PyTorch: {torch.__version__}")
log.info(f"Platform: {hw['platform']}")
for k, v in hw.items():
    print(f"  {k}: {v}")

if torch.cuda.is_available():
    log.info(f"GPU: {hw.get('gpu', 'N/A')}")
    log.info(f"VRAM: {hw.get('gpu_memory', 'N/A')}")
    log.info(f"CUDA: {hw.get('cuda', 'N/A')}")

# ── Load HuggingFace token ──
HF_TOKEN = _resolve_hf_token()
if HF_TOKEN:
    log.success("HuggingFace token loaded")
    print("  ✓ HF token loaded")
else:
    log.warning("No HuggingFace token — checkpoint upload will be skipped")
    print("  ⚠ No HF token — add HF_TOKEN to Kaggle Secrets for auto-upload")

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

---
## Cell 4: Load Phase 1 & Phase 2 Checkpoints + Smoke Test

Verifies the checkpoint chain before Phase 4 builds on top.
- **Phase 1 (CSE):** Encodes text → semantic wave [seq_len, 432]
- **Phase 2 (Field):** Stores patterns as energy attractors in 3D field
- **Phase 4 (TL):** Will wire thermodynamic settling onto the field

In [ ]:
log.cell_start("Cell 4 — Load Phase 1 & Phase 2 + Smoke Test")

import torch
import torch.nn.functional as F

# Force-reimport phase modules
import importlib
for _m in ['cse', 'field', 'thermodynamic', 'temperature', 'energy_functions', 'online_learner']:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

from cse import ContinuousSemanticEncoder
from field import ResonanceField, FIELD_H, FIELD_W, FIELD_D, FIELD_FEATURES
from thermodynamic import ThermodynamicLearner
from online_learner import OnlineLearner

# ── Load Phase 1 (CSE) ──
ckpt1 = load_checkpoint(1)
cse = ContinuousSemanticEncoder(**ckpt1.get('config', {}))
cse.load_state_dict(ckpt1['state_dict'])
cse = cse.to(DEVICE).eval()
for p in cse.parameters():
    p.requires_grad = False

# Smoke test CSE
test_wave = cse.encode("smoke test Phase 1")
assert test_wave.full.shape[-1] == 432, f"Bad wave dim: {test_wave.full.shape}"
assert not torch.isnan(test_wave.full).any(), "NaN in wave!"
log.success(f"Phase 1 CSE loaded: {sum(p.numel() for p in cse.parameters()):,} params")
print(f"  ✓ Phase 1 (CSE): wave shape {test_wave.full.shape}")

# ── Load Phase 2 (ResonanceField) ──
ckpt2 = load_checkpoint(2)
field_cfg = ckpt2.get('config', {}).get('field', {})
field = ResonanceField(**field_cfg)
field.load_state_dict(ckpt2['state_dict'])
field = field.to(DEVICE)

# Smoke test Field
vec = test_wave.full.mean(dim=0).to(DEVICE)
field_out, sims, locs = field.query(vec, k=4)
assert not torch.isnan(field_out).any(), "NaN in field query!"
log.success(f"Phase 2 Field loaded: {sum(p.numel() for p in field.parameters()):,} params")
print(f"  ✓ Phase 2 (ResonanceField): field {FIELD_H}×{FIELD_W}×{FIELD_D}×{FIELD_FEATURES}")

# ── Smoke test ThermodynamicLearner ──
tl_test = ThermodynamicLearner(field=field, settle_iterations=2).to(DEVICE)
test_result = tl_test.settle_once(vec)
assert test_result.final_energy >= 0, "Negative energy!"
log.success(f"ThermodynamicLearner smoke test: energy {test_result.initial_energy:.4f} → {test_result.final_energy:.4f}")
print(f"  ✓ Phase 4 (TL): settle OK — energy {test_result.initial_energy:.4f} → {test_result.final_energy:.4f}")

# ── Smoke test OnlineLearner ──
ol_test = OnlineLearner(cse=cse, tl=tl_test, device=DEVICE)
ol_result = ol_test.learn_fact("the sky is blue")
log.success(f"OnlineLearner smoke test: fact_stored={ol_result.fact_stored}")
print(f"  ✓ OnlineLearner: learn_fact OK — stored={ol_result.fact_stored}")

del tl_test, ol_test
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("\n  Phase 1 model: https://huggingface.co/UnseenGAP/FLUX (phase1.phase.pt)")
print("  Phase 2 model: https://huggingface.co/UnseenGAP/FLUX (phase2.phase.pt)")
log.cell_end("Cell 4 — Load Phase 1 & Phase 2 + Smoke Test", "PASS")

---
## Cell 5: Training / Load from Checkpoint

> **Smart skip:** If a Phase 4 checkpoint already exists locally or on HuggingFace Hub, this cell loads it and skips training.

**If training from scratch — five stages:**
- **Stage A:** One-shot fact learning (10 facts, 1 settle pass each)
- **Stage B:** Retention test (learn fact, feed 100 distractors, verify retrieval)
- **Stage C:** Temperature annealing observation over 200 steps
- **Stage D:** Compare thermodynamic learning vs SGD convergence
- **Stage E:** Verify zero global gradients

~5 min on GPU • ~20 min on CPU

In [ ]:
log.cell_start("Cell 5 — Training / Load from Checkpoint")

import time
import torch
import torch.nn.functional as F
from thermodynamic import ThermodynamicLearner
from online_learner import OnlineLearner

# ─────────────────────────────────────────────────────────────────
# SKIP GUARD — if Phase 4 checkpoint already exists, load it
# directly and skip all training stages.
# ─────────────────────────────────────────────────────────────────
_PHASE4_FRESH_TRAIN = not checkpoint_exists(4)

if not _PHASE4_FRESH_TRAIN:
    print("  ✓ Phase 4 checkpoint found — loading saved model (skipping training)")
    log.info("Phase 4 checkpoint exists — loading instead of training")

    ckpt4 = load_checkpoint(4)
    tl = ThermodynamicLearner(field=field, settle_iterations=10, decay=0.995).to(DEVICE)
    tl.load_state(ckpt4['tl_state'])
    ol = OnlineLearner(cse=cse, tl=tl, device=DEVICE)

    _stored_metrics = ckpt4.get('metrics', {})
    elapsed = _stored_metrics.get('training_time_seconds', 0.0)

    print(f"  ✓ Checkpoint loaded")
    print(f"  ✓ Facts stored:      {_stored_metrics.get('facts_stored', 'N/A')}")
    print(f"  ✓ Retention sim:     {_stored_metrics.get('retention_sim', 'N/A')}")
    print(f"  ✓ Final temperature: {_stored_metrics.get('final_temperature', 'N/A')}")
    print("\n  ⏩ Skipping training — continuing straight to tests, demos & results")

else:
    # ── FULL TRAINING PATH ────────────────────────────────────────
    start_time = time.time()

    # ── Build Phase 4 components ──
    # Reload field from Phase 2 (clean copy for training)
    field = ResonanceField(**field_cfg)
    field.load_state_dict(ckpt2['state_dict'])
    field = field.to(DEVICE)

    tl = ThermodynamicLearner(
        field=field,
        initial_temp=1.0,
        min_temp=0.01,
        max_temp=2.0,
        decay=0.995,
        settle_iterations=10,
        settle_dt=0.1,
        perturbation_radius=4,
        error_sensitivity=0.5,
    ).to(DEVICE)

    ol = OnlineLearner(cse=cse, tl=tl, device=DEVICE)
    log.success("ThermodynamicLearner + OnlineLearner built")

    # ════════════════════════════════════════════
    # Stage A: One-Shot Fact Learning
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage A: One-Shot Fact Learning")
    print("=" * 65)

    test_facts = [
        "The capital of Mars colony Alpha is New Houston",
        "Water boils at 100 degrees Celsius at sea level",
        "The speed of light is approximately 300000 km per second",
        "Photosynthesis converts carbon dioxide into oxygen",
        "The deepest ocean trench is the Mariana Trench",
        "DNA carries genetic information in all living organisms",
        "The Earth orbits the Sun once every 365 days",
        "Gravity on the Moon is one sixth of Earth gravity",
        "The human brain contains approximately 86 billion neurons",
        "Sound travels faster in water than in air",
    ]

    fact_results = []
    for i, fact in enumerate(test_facts):
        result = ol.learn_fact(fact)
        fact_results.append(result)
        icon = "✓" if result.fact_stored else "✗"
        print(f"  {icon} [{i+1:2d}] energy: {result.initial_energy:.4f} → {result.final_energy:.4f} "
              f"(Δ={result.energy_delta:+.4f})  temp={result.temperature:.4f}  "
              f"surprise={result.prediction_error:.4f}")

    stored_count = sum(r.fact_stored for r in fact_results)
    log.metric("facts_stored", f"{stored_count}/{len(test_facts)}")
    print(f"\n  Facts stored (energy decreased): {stored_count}/{len(test_facts)}")

    # ════════════════════════════════════════════
    # Stage B: Retention Test
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage B: Retention After 100 Distractor Updates")
    print("=" * 65)

    distractors = [
        "The weather today is partly cloudy with a chance of rain",
        "Apples are a popular fruit grown in temperate climates",
        "Chess is a strategic board game played by two opponents",
        "Mount Everest is the tallest mountain above sea level",
        "The Pacific Ocean is the largest ocean on Earth",
        "Electricity flows through conductors like copper wire",
        "Volcanoes form at tectonic plate boundaries",
        "The Sahara is the largest hot desert in the world",
        "Penguins are flightless birds found in the southern hemisphere",
        "The Amazon rainforest produces 20 percent of the world oxygen",
    ] * 12

    retention = ol.test_retention(
        fact_text="The capital of Mars colony Alpha is New Houston",
        distractor_texts=distractors,
        n_distractors=100,
    )
    print(f"  Fact: '{retention['fact']}'")
    print(f"  Similarity immediately after learning: {retention['sim_immediately']:.4f}")
    print(f"  Similarity after {retention['n_distractors']} distractors: {retention['sim_after_distractors']:.4f}")
    print(f"  Similarity drop: {retention['similarity_drop']:.4f}")
    print(f"  Retained: {retention['retained']}")
    log.metric("retention_sim_immediately", f"{retention['sim_immediately']:.4f}")
    log.metric("retention_sim_after_100", f"{retention['sim_after_distractors']:.4f}")
    log.metric("retention_passed", str(retention['retained']))

    # ════════════════════════════════════════════
    # Stage C: Temperature Annealing Observation
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage C: Temperature Annealing Over 200 Steps")
    print("=" * 65)

    stream_texts = distractors[:200]
    stream_pairs = []
    for text in stream_texts:
        wave_vec = ol._text_to_wave(text)
        stream_pairs.append((wave_vec, None))

    stream_results = tl.learn_stream(stream_pairs, verbose=True)

    temp_stats = tl.temp_manager.stats()
    print(f"\n  Final temperature:    {temp_stats['temperature']:.6f}")
    print(f"  Steps completed:      {temp_stats['step_count']}")
    print(f"  Is cold:              {temp_stats['is_cold']}")
    if 'error_trend' in temp_stats:
        print(f"  Error trend:          {temp_stats['error_trend']:+.6f}")
    log.metric("final_temperature", f"{temp_stats['temperature']:.6f}")

    # ════════════════════════════════════════════
    # Stage D: SGD Comparison
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage D: Thermodynamic Learning vs SGD Comparison")
    print("=" * 65)

    comparison = ol.compare_to_sgd(test_facts[:5], sgd_steps=100)
    print(f"  Facts compared:       {comparison['n_facts']}")
    print(f"  TL time:              {comparison['tl_time']:.3f}s")
    print(f"  SGD time:             {comparison['sgd_time']:.3f}s")
    print(f"  TL mean energy:       {comparison['tl_mean_energy']:.6f}")
    print(f"  SGD mean energy:      {comparison['sgd_mean_energy']:.6f}")
    print(f"  TL faster:            {comparison['tl_faster']}")
    print(f"  Speedup factor:       {comparison['speedup']:.2f}x")
    log.metric("tl_vs_sgd_speedup", f"{comparison['speedup']:.2f}x")

    # ════════════════════════════════════════════
    # Stage E: Verify No Global Gradients
    # ════════════════════════════════════════════
    print("\n" + "=" * 65)
    print("  Stage E: Verify No Global Gradients")
    print("=" * 65)

    has_grads = False
    for name, param in field.named_parameters():
        if param.grad is not None and param.grad.abs().sum() > 0:
            has_grads = True
            print(f"  ✗ UNEXPECTED gradient in {name}: {param.grad.abs().sum():.6f}")

    if not has_grads:
        print("  ✓ No global gradients found in field parameters")
        print("  ✓ All updates were local (through physics, not backprop)")
    log.metric("no_global_gradients", str(not has_grads))

    elapsed = time.time() - start_time
    log.metric("training_time", f"{elapsed:.1f}s")
    print(f"\n  ✓ Training complete in {elapsed:.1f}s ({elapsed/60:.1f} min)")

log.cell_end("Cell 5 — Training / Load from Checkpoint", "PASS")

---
## Cell 6: Save Checkpoint & Upload to HuggingFace Hub

> **Smart skip:** If checkpoint already existed, this cell skips save and upload.

In [ ]:
log.cell_start("Cell 6 — Save & Upload Checkpoint")

from datetime import datetime

if not _PHASE4_FRESH_TRAIN:
    ckpt_path = Path('checkpoints/phase4.phase.pt')
    size_mb = ckpt_path.stat().st_size / 1e6 if ckpt_path.exists() else 0
    print(f"  ⏩ Checkpoint already saved — skipping save step")
    print(f"     Local:  {ckpt_path} ({size_mb:.1f} MB)")
    print(f"     Remote: https://huggingface.co/UnseenGAP/FLUX")
    log.info("Checkpoint already existed — save step skipped")
else:
    checkpoint_state = {
        'phase': 4,
        'timestamp': datetime.now().isoformat(),
        'phase1_config': ckpt1.get('config', {}),
        'phase2_config': ckpt2.get('config', {}),
        'tl_state': tl.save_state(),
        'metrics': {
            'facts_stored': stored_count,
            'retention_sim': retention['sim_after_distractors'],
            'retention_passed': retention['retained'],
            'final_temperature': tl.temp_manager.temperature,
            'no_global_gradients': not has_grads,
            'tl_faster_than_sgd': comparison['tl_faster'],
            'sgd_speedup': comparison['speedup'],
            'training_time_seconds': elapsed,
        },
    }
    save_checkpoint(4, checkpoint_state)

    ckpt_path = Path('checkpoints/phase4.phase.pt')
    if ckpt_path.exists():
        size_mb = ckpt_path.stat().st_size / 1e6
        log.success(f"Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
        print(f"  ✓ Checkpoint saved: {ckpt_path} ({size_mb:.1f} MB)")
    else:
        log.error("Checkpoint NOT found — save may have failed")

    uploaded = upload_checkpoint_to_hf(phase=4, hf_token=HF_TOKEN)
    if uploaded:
        log.success("Checkpoint uploaded to https://huggingface.co/UnseenGAP/FLUX")
        print("  ✓ Uploaded to HuggingFace Hub")
    else:
        log.warning("Checkpoint upload skipped — check HF_TOKEN")
        print("  ⚠ Upload skipped — no HF token")

    upload_logs_to_hf(phase=4, hf_token=HF_TOKEN)

log.cell_end("Cell 6 — Save & Upload Checkpoint", "PASS")

---
## Cells 7–9: Tests

| Test | Description | Pass Criteria |
|------|-------------|---------------|
| 1 | Single-shot learning retention | Fact survives 100+ updates, sim > 0.5 |
| 2 | No global gradient required | Zero .grad on all field parameters |
| 3 | Temperature annealing behavior | Correct heating/cooling dynamics |

In [ ]:
log.cell_start("Cell 7 — Test 1: Single-Shot Learning Retention")

os.chdir(str(WORK_DIR / 'phases' / 'phase4'))
%run test_phase4_test1.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 7 — Test 1: Single-Shot Learning Retention", "PASS")

In [ ]:
log.cell_start("Cell 8 — Test 2: No Global Gradient Required")

os.chdir(str(WORK_DIR / 'phases' / 'phase4'))
%run test_phase4_test2.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 8 — Test 2: No Global Gradient Required", "PASS")

In [ ]:
log.cell_start("Cell 9 — Test 3: Temperature Annealing Behavior")

os.chdir(str(WORK_DIR / 'phases' / 'phase4'))
%run test_phase4_test3.py
os.chdir(str(WORK_DIR))

log.cell_end("Cell 9 — Test 3: Temperature Annealing Behavior", "PASS")

---
## Cells 10–11: Demos

| Demo | Description |
|------|-------------|
| 1 | One-shot fact learning — energy drops, temperature annealing, retrieval similarity |
| 2 | Thermodynamic learning vs SGD — convergence speed, gradient count, energy comparison |

In [ ]:
log.cell_start("Cell 10 — Demo 1: One-Shot Fact Learning")

os.chdir(str(WORK_DIR / 'phases' / 'phase4'))
%run demo_phase4_demo1.py

from IPython.display import Image, display
img_path = WORK_DIR / 'phases' / 'phase4' / 'demo4_oneshot_learning.png'
if img_path.exists():
    display(Image(filename=str(img_path), width=900))
    log.success("One-shot learning chart generated")
else:
    log.warning("One-shot chart not generated")

os.chdir(str(WORK_DIR))
log.cell_end("Cell 10 — Demo 1: One-Shot Fact Learning", "PASS")

In [ ]:
log.cell_start("Cell 11 — Demo 2: TL vs SGD Comparison")

os.chdir(str(WORK_DIR / 'phases' / 'phase4'))
%run demo_phase4_demo2.py

from IPython.display import Image, display
img_path = WORK_DIR / 'phases' / 'phase4' / 'demo4_tl_vs_sgd.png'
if img_path.exists():
    display(Image(filename=str(img_path), width=900))
    log.success("TL vs SGD chart generated")
else:
    log.warning("TL vs SGD chart not generated")

os.chdir(str(WORK_DIR))
log.cell_end("Cell 11 — Demo 2: TL vs SGD Comparison", "PASS")

---
## Cell 12: Interactive Exploration
Learn custom facts on the fly, query them back, and observe temperature dynamics.

In [ ]:
log.cell_start("Cell 12 — Interactive Exploration")

import torch
import torch.nn.functional as F
from thermodynamic import ThermodynamicLearner
from online_learner import OnlineLearner

# ── Reload from saved checkpoint ──
ckpt4 = load_checkpoint(4)
field_interactive = ResonanceField(**field_cfg)
field_interactive.load_state_dict(ckpt2['state_dict'])
field_interactive = field_interactive.to('cpu')

tl_interactive = ThermodynamicLearner(field=field_interactive, settle_iterations=10).to('cpu')
tl_interactive.load_state(ckpt4['tl_state'])
ol_interactive = OnlineLearner(cse=cse.to('cpu'), tl=tl_interactive, device='cpu')

print("  Thermodynamic Learner Statistics:")
print("  " + "-" * 45)
stats = tl_interactive.stats()
for k, v in stats.items():
    if isinstance(v, float):
        print(f"    {k:<25}: {v:.4f}")
    else:
        print(f"    {k:<25}: {v}")
    log.metric(k, str(v))

# ── Learn custom facts ──
print("\n  Learning Custom Facts:")
print("  " + "-" * 72)
custom_facts = [
    "FLUX replaces backpropagation with thermodynamic settling",
    "The Andromeda galaxy is 2.5 million light years away",
    "Python is a popular programming language",
    "Mozart composed his first symphony at age eight",
    "The speed of sound in air is 343 meters per second",
]

for fact in custom_facts:
    result = ol_interactive.learn_fact(fact)
    icon = "✓" if result.fact_stored else "✗"
    print(f"  {icon} '{fact[:50]}...'  energy: {result.final_energy:.4f}  temp: {result.temperature:.6f}")

# ── Query them back ──
print("\n  Querying Facts Back:")
print("  " + "-" * 72)
print(f"  {'Query':<50} {'Similarity':>10} {'Energy':>10}")
print("  " + "-" * 72)

for fact in custom_facts:
    qr = ol_interactive.query_fact(fact)
    print(f"  {fact[:50]:<50} {qr.top_similarity:>10.4f} {qr.energy_at_location:>10.4f}")
    log.metric(f"explore('{fact[:20]}')", f"sim={qr.top_similarity:.4f}")

# ── Temperature after exploration ──
print(f"\n  Current temperature: {tl_interactive.temp_manager.temperature:.6f}")
print(f"  Total facts learned: {ol_interactive.learned_count}")

print("\n  ✓ Interactive exploration complete")

# Restore to GPU
cse = cse.to(DEVICE)
field = field.to(DEVICE)
del field_interactive, tl_interactive, ol_interactive

log.cell_end("Cell 12 — Interactive Exploration", "PASS")

---
## Cell 13: View Results Summary & Full Log

In [ ]:
log.cell_start("Cell 13 — Results Summary & Log")

from IPython.display import Markdown, display

results_path = WORK_DIR / 'phases' / 'phase4' / 'RESULTS_PHASE_4.md'
if results_path.exists():
    display(Markdown(results_path.read_text()))
    log.success("Results displayed")
else:
    print("  ⚠ RESULTS_PHASE_4.md not yet generated — run tests first")
    print("  Tests auto-generate this file via PhaseResults utility.")
    log.warning("No RESULTS_PHASE_4.md found")

print("\n" + "=" * 60)
print("FULL PHASE 4 LOG")
print("=" * 60)
print(log.get_log_contents())

log.cell_end("Cell 13 — Results Summary & Log", "PASS")

---
## Cell 14: Final Upload — Logs to HuggingFace & GitHub
Pushes the complete Phase 4 log, results, and checkpoint to both platforms.

In [ ]:
log.cell_start("Cell 14 — Final Upload")

upload_logs_to_hf(phase=4, hf_token=HF_TOKEN)
log.success("Logs uploaded to HuggingFace Hub")

git_commit_and_push(
    files=[
        'logs/phase4.log',
        'results/',
        'phases/phase4/RESULTS_PHASE_4.md',
    ],
    message='Phase 4: Thermodynamic Learning — training complete [auto-commit from Kaggle]',
    repo_dir=str(WORK_DIR),
)

log.cell_end("Cell 14 — Final Upload", "PASS")

print("\n" + "=" * 60)
print("✓ PHASE 4 COMPLETE")
print("=" * 60)
print(f"  Checkpoint: https://huggingface.co/UnseenGAP/FLUX")
print(f"  Logs:       https://huggingface.co/UnseenGAP/FLUX (logs/)")
print(f"  Code:       https://github.com/Unseengap/FLUX")
print(f"  Next:       Phase 5 — Causal Geometry Nodes")
print("=" * 60)

---
## Cell 15: Save Artifacts to Kaggle Output

In [ ]:
log.cell_start("Cell 15 — Save Kaggle Artifacts")

import shutil

output_dir = Path('/kaggle/working/output')
output_dir.mkdir(exist_ok=True)

# ── Checkpoints ──
for fname in ['phase4.phase.pt']:
    src = WORK_DIR / 'checkpoints' / fname
    if src.exists():
        shutil.copy2(str(src), str(output_dir / src.name))
        print(f"  ✓ Checkpoint: {src.name} ({src.stat().st_size/1e6:.1f} MB)")

# ── Results and logs ──
for fpath in [
    WORK_DIR / 'phases' / 'phase4' / 'RESULTS_PHASE_4.md',
    WORK_DIR / 'logs' / 'phase4.log',
]:
    if fpath.exists():
        shutil.copy2(str(fpath), str(output_dir / fpath.name))
        print(f"  ✓ {fpath.name}")

# ── Demo images ──
for img in (WORK_DIR / 'phases' / 'phase4').glob('*.png'):
    shutil.copy2(str(img), str(output_dir / img.name))
    print(f"  ✓ {img.name}")

print(f"\n  Kaggle output artifacts:")
for f in sorted(output_dir.iterdir()):
    print(f"    {f.name} ({f.stat().st_size / 1e6:.2f} MB)")

log.cell_end("Cell 15 — Save Kaggle Artifacts", "PASS")

print("\n" + "=" * 60)
print("  PHASE 4: Thermodynamic Learning — ALL DONE ✓")
print("=" * 60)

---
## Phase 4 Acceptance Criteria

| Criteria | Cell | Target |
|---|---|---|
| Model learns from single example (no batch, no epochs) | Stage A / Test 1 | ≥ 50% facts stored |
| Learned fact retrievable after 100 subsequent updates | Stage B / Test 1 | sim > 0.5 |
| No global gradient computation at any point | Stage E / Test 2 | Zero .grad |
| Temperature decreases as error decreases | Stage C / Test 3 | monotonic on low error |
| Learning faster than equivalent SGD on simple tasks | Stage D / Demo 2 | TL faster or comparable |
| All tests pass | Tests 1–3 | ✓ |
| Phase 1 checkpoint loads correctly | Cell 4 | ✓ |
| Phase 2 checkpoint loads correctly | Cell 4 | ✓ |
| Phase 4 checkpoint saved | Cell 6 | `checkpoints/phase4.phase.pt` |
| Checkpoint uploaded to HuggingFace | Cell 6 | `UnseenGAP/FLUX` |
| Logs written to logs/phase4.log | All cells | ✓ |
| RESULTS_PHASE_4.md generated | Tests | ✓ |

### Key Design Decisions
- **No loss.backward()** — all field updates through `field.perturb()` + `field.settle()`
- **Temperature is local** — surprise heats the system, confidence cools it
- **No epochs** — each sample seen exactly once, in order, as a stream
- **Retention via mass** — established attractors resist change (high-mass = high inertia)
- **Inference = learning** — `settle_once()` is both at the same time